In [5]:
# ==========================================
# CELL 1: Data Loading, Splitting & Preprocessing
# ==========================================
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# 1. Define 25 Predictors & Target Tier
predictors = [
    'AQI_lag24', 'Raw_Conc_lag24', 'Hour', 'Month', 'DayOfWeek',
    'temperature_2m', 'relativehumidity_2m', 'dewpoint_2m', 'apparent_temperature',
    'precipitation', 'rain', 'surface_pressure', 'cloudcover', 'cloudcover_low',
    'cloudcover_mid', 'cloudcover_high', 'windspeed_10m', 'winddirection_10m',
    'windgusts_10m', 'et0_fao_evapotranspiration', 'vapor_pressure_deficit',
    'shortwave_radiation', 'direct_radiation', 'diffuse_radiation', 'weathercode'
]
target = 'Hazard_Tier'
seasons = ['winter', 'summer', 'monsoon', 'post_monsoon']

processed_data = {}

print("--- [CELL 1] Loading and Preprocessing Seasonal Datasets ---")

for season in seasons:
    possible_names = [f"{season}.csv", f"data_{season}.csv", f"dhaka_{season}.csv"]
    file_path = None

    for name in possible_names:
        if os.path.exists(name):
            file_path = name
            break

    if file_path is None:
        raise FileNotFoundError(f"Could not find CSV for '{season}'. Expected one of: {possible_names}")

    df = pd.read_csv(file_path)

    # Time-based 80/20 train-test split
    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]

    X_train_raw = train_df[predictors]
    y_train_raw = train_df[target]
    X_test_raw = test_df[predictors]
    y_test_raw = test_df[target]

    # Scale Features (GaussianNB assumes continuous normally distributed features)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # Apply SMOTE
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train_raw)

    processed_data[season] = {
        'X_train': X_train_res,
        'y_train': y_train_res,
        'X_test': X_test_scaled,
        'y_test': y_test_raw,
        'scaler': scaler
    }

    print(f"[{season.capitalize():<12}] Loaded '{file_path}' | Train: {X_train_res.shape[0]} samples (SMOTE) | Test: {X_test_scaled.shape[0]} samples")

print("\nData preparation complete.\n")

--- [CELL 1] Loading and Preprocessing Seasonal Datasets ---
[Winter      ] Loaded 'data_winter.csv' | Train: 17775 samples (SMOTE) | Test: 2529 samples
[Summer      ] Loaded 'data_summer.csv' | Train: 26604 samples (SMOTE) | Test: 2928 samples
[Monsoon     ] Loaded 'data_monsoon.csv' | Train: 24753 samples (SMOTE) | Test: 3175 samples
[Post_monsoon] Loaded 'data_post_monsoon.csv' | Train: 13272 samples (SMOTE) | Test: 1560 samples

Data preparation complete.



In [6]:
# ==========================================
# CELL 2: Training Gaussian Naive Bayes Models Across Seasons
# ==========================================
from sklearn.naive_bayes import GaussianNB

trained_models = {}

print("--- [CELL 2] Training Gaussian Naive Bayes Models Across Seasons ---")

for season in seasons:
    print(f"\nTraining Naive Bayes for Season: [{season.upper()}]...")

    X_tr = processed_data[season]['X_train']
    y_tr = processed_data[season]['y_train']

    # Instantiate Gaussian Naive Bayes classifier
    nb_model = GaussianNB()

    nb_model.fit(X_tr, y_tr)
    trained_models[season] = nb_model

    print(f"  --> Successfully trained GaussianNB for {season.capitalize()}.")

print("\nModel training phase complete.")

--- [CELL 2] Training Gaussian Naive Bayes Models Across Seasons ---

Training Naive Bayes for Season: [WINTER]...
  --> Successfully trained GaussianNB for Winter.

Training Naive Bayes for Season: [SUMMER]...
  --> Successfully trained GaussianNB for Summer.

Training Naive Bayes for Season: [MONSOON]...
  --> Successfully trained GaussianNB for Monsoon.

Training Naive Bayes for Season: [POST_MONSOON]...
  --> Successfully trained GaussianNB for Post_monsoon.

Model training phase complete.


In [7]:
# ==========================================
# CELL 3: Model Evaluation & Automated CSV Export
# ==========================================
import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix

def calc_gmean(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True)
    recalls = [report[cls]['recall'] for cls in report if cls not in ['accuracy', 'macro avg', 'weighted avg']]
    return np.exp(np.mean(np.log(np.maximum(recalls, 1e-5))))

results = []
detailed_reports = []
evaluation_logs = {}

print("--- [CELL 3] Evaluating Within-Season Test Performance ---")

for season in seasons:
    model = trained_models[season]
    X_te = processed_data[season]['X_test']
    y_te = processed_data[season]['y_test']

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)

    macro_f1 = f1_score(y_te, y_pred, average='macro')
    g_mean = calc_gmean(y_te, y_pred)

    try:
        macro_auc = roc_auc_score(y_te, y_proba, multi_class='ovr', average='macro')
    except ValueError:
        macro_auc = np.nan

    cm = confusion_matrix(y_te, y_pred)
    report_dict = classification_report(y_te, y_pred, output_dict=True)

    evaluation_logs[season] = {
        'y_true': y_te,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'cm': cm,
        'report': classification_report(y_te, y_pred)
    }

    results.append({
        'Model': 'Naive Bayes',
        'Season': season.capitalize(),
        'Macro F1': round(macro_f1, 4),
        'Macro AUC': round(macro_auc, 4),
        'G-Mean': round(g_mean, 4)
    })

    for cls_name, metrics in report_dict.items():
        if isinstance(metrics, dict):
            detailed_reports.append({
                'Season': season.capitalize(),
                'Class': cls_name,
                'Precision': round(metrics['precision'], 4),
                'Recall': round(metrics['recall'], 4),
                'F1-Score': round(metrics['f1-score'], 4),
                'Support': int(metrics['support'])
            })

# Export CSV Tables
tables_dir = os.path.join("results", "tables")
os.makedirs(tables_dir, exist_ok=True)

results_df = pd.DataFrame(results)
summary_csv_path = os.path.join(tables_dir, "naive_bayes_e1.csv")
results_df.to_csv(summary_csv_path, index=False)

detailed_df = pd.DataFrame(detailed_reports)
detailed_csv_path = os.path.join(tables_dir, "naive_bayes_e1_detailed.csv")
detailed_df.to_csv(detailed_csv_path, index=False)

print("\n=== Naive Bayes: Within-Season Baseline (E1) ===")
print(results_df.to_string(index=False))

print(f"\n[SUCCESS] Exported summary table to: {summary_csv_path}")
print(f"[SUCCESS] Exported detailed classification table to: {detailed_csv_path}")

--- [CELL 3] Evaluating Within-Season Test Performance ---

=== Naive Bayes: Within-Season Baseline (E1) ===
      Model       Season  Macro F1  Macro AUC  G-Mean
Naive Bayes       Winter    0.5609     0.7911  0.7173
Naive Bayes       Summer    0.4238     0.7999  0.5927
Naive Bayes      Monsoon    0.3939     0.7231  0.5497
Naive Bayes Post_monsoon    0.5481     0.9144  0.7630

[SUCCESS] Exported summary table to: results/tables/naive_bayes_e1.csv
[SUCCESS] Exported detailed classification table to: results/tables/naive_bayes_e1_detailed.csv


In [ ]:
# ==========================================
# CELL 4: Individual Slide Graphs & Auto-Download
# ==========================================
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance
from google.colab import files

print("--- [CELL 4] Generating Individual Slide Figures & Downloading ---")

output_dir = os.path.join("results", "visualizations", "naive_bayes")
os.makedirs(output_dir, exist_ok=True)

download_queue = []

# 1. Save Confusion Matrices per Season
for season in seasons:
    plt.figure(figsize=(6, 5))
    cm = evaluation_logs[season]['cm']
    labels = sorted(list(set(evaluation_logs[season]['y_true'])))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        cbar=False,
        annot_kws={"size": 12, "weight": "bold"}
    )
    plt.title(f'Confusion Matrix: {season.capitalize()} (NB)', fontsize=14, fontweight='bold', pad=12)
    plt.xlabel('Predicted Tier', fontsize=11, fontweight='bold')
    plt.ylabel('True Tier', fontsize=11, fontweight='bold')
    plt.tight_layout()

    file_path = os.path.join(output_dir, f"cm_{season}.png")
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.show()
    download_queue.append(file_path)

# 2. Save Permutation Feature Importance Graphic (Winter)
plt.figure(figsize=(8, 5))

winter_nb = trained_models['winter']
X_te_winter = processed_data['winter']['X_test']
y_te_winter = processed_data['winter']['y_test']

# Compute model-agnostic Permutation Importance for Naive Bayes
perm_importance = permutation_importance(winter_nb, X_te_winter, y_te_winter, scoring='f1_macro', n_repeats=10, random_state=42)
mean_importances = perm_importance.importances_mean
top_idx = np.argsort(mean_importances)[-10:]

bars = plt.barh(np.array(predictors)[top_idx], np.maximum(mean_importances[top_idx], 0), color='#6a1b9a', edgecolor='black')
plt.title('Top 10 Predictor Importance (Naive Bayes - Winter)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Mean Decrease in Macro F1 Score', fontsize=11, fontweight='bold')
plt.ylabel('Predictor Feature', fontsize=11, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.7)

for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.001, bar.get_y() + bar.get_height()/2, f'{width:.3f}',
             va='center', ha='left', fontsize=9, fontweight='bold')

plt.tight_layout()

feat_file_path = os.path.join(output_dir, "feature_importance_winter.png")
plt.savefig(feat_file_path, dpi=300, bbox_inches='tight')
plt.show()
download_queue.append(feat_file_path)

# 3. Trigger Browser Downloads
print("\n--- Triggering Automatic Browser Downloads ---")
for file_path in download_queue:
    print(f"Downloading: {file_path}")
    files.download(file_path)